In [ ]:
# -----------------------------
# Import libraries
# -----------------------------
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import xgboost as xgb
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.preprocessing import StandardScaler 
from sklearn.datasets import make_regression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_curve, auc
from sklearn.metrics import make_scorer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from string import ascii_uppercase
from geopy.distance import geodesic
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn import linear_model
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import numpy as np
from xgboost import XGBRegressor
from tqdm import tqdm
import time
from scipy.stats import uniform
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

In [ ]:
# -----------------------------
# Import local functions
# -----------------------------
import sys
import os

# Get the absolute path to your src folder
module_path = os.path.abspath(os.path.join('..', 'src'))

if module_path not in sys.path:
    sys.path.append(module_path)

from cleaning_and_helpers import plot_test_preds, split_project, multioutput_rmse, evaluate_model


In [ ]:
# -----------------------------
# Set random seed, also for train test split set and models set seed within function
# -----------------------------
np.random.seed(1298)
seed = 1298

In [ ]:
# -----------------------------
# Load .npy files with features and targets from each project
# -----------------------------

SI_X = np.load('../../pollenGeolocation_saved/SI_X_raw.npy')
SI_y = np.load('../../pollenGeolocation_saved/SI_y.npy')

FFAR_X = np.load('../../pollenGeolocation_saved/FFAR_X_raw.npy')
FFAR_y = np.load('../../pollenGeolocation_saved/FFAR_y.npy')

NCASI_X = np.load('../../pollenGeolocation_saved/NCASI_X_raw.npy')
NCASI_y = np.load('../../pollenGeolocation_saved/NCASI_y.npy')

In [ ]:
# -----------------------------
# Train test split for each project
# ----------------------------- 
test_size = 0.20

# -----------------------------
# Split each project
# -----------------------------
SI_X_train, SI_X_test, SI_y_train, SI_y_test = split_project(SI_X, SI_y, test_size, seed)
NCASI_X_train, NCASI_X_test, NCASI_y_train, NCASI_y_test = split_project(NCASI_X, NCASI_y, test_size, seed)
FFAR_X_train, FFAR_X_test, FFAR_y_train, FFAR_y_test = split_project(FFAR_X, FFAR_y, test_size, seed)

# -----------------------------
# Concatenate train and test splits from all projects
# -----------------------------
X_train = np.concatenate([SI_X_train, NCASI_X_train, FFAR_X_train], axis=0)
y_train = np.concatenate([SI_y_train, NCASI_y_train, FFAR_y_train], axis=0)

X_test = np.concatenate([SI_X_test, NCASI_X_test, FFAR_X_test], axis=0)
y_test = np.concatenate([SI_y_test, NCASI_y_test, FFAR_y_test], axis=0)

# -----------------------------
# Standardize X and y using separate scalers for training and test set to prevent information
# leakage
# -----------------------------
sc_X = StandardScaler()
sc_y = StandardScaler()

X_train = sc_X.fit_transform(X_train)
y_train = sc_y.fit_transform(y_train)

X_test = sc_X.transform(X_test)
y_test = sc_y.transform(y_test)


In [ ]:
# -----------------------------
# Custom RMSE scorer
# -----------------------------

rmse_scorer = make_scorer(multioutput_rmse, greater_is_better=False)

# -----------------------------
# Cross-validation setup
# -----------------------------
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=1298)

# -----------------------------
# Models and param grids
# -----------------------------
models = {
    "MultiTaskLasso": (
        linear_model.MultiTaskLasso(max_iter=10000),
        {
            'estimator__alpha': np.arange(0.00, 1.0, 0.01)
        }
    ),

    "SVR": (
        MultiOutputRegressor(SVR()),
        {
            'estimator__estimator__C': uniform(0.1, 10),  # Uniform distribution between 0.1 and 10
            'estimator__estimator__kernel': ['linear', 'rbf', 'poly'],
            'estimator__estimator__gamma': ['scale', 'auto']
        }  
    ),

    "KNN": (
        MultiOutputRegressor(KNeighborsRegressor()),
        {
            'estimator__estimator__n_neighbors': np.arange(2, 50, 1),  # Number of neighbors (2 to 50)
            'estimator__estimator__weights': ['uniform', 'distance'],  # Weighting method
            'estimator__estimator__p': [1, 2],  # Manhattan or Euclidean distance
            'estimator__estimator__algorithm': ['auto']  # Default to 'auto'
        }
    ),

    "DecisionTree": (
        MultiOutputRegressor(DecisionTreeRegressor(random_state=seed)),
        {
            'estimator__estimator__max_depth': [None] + list(range(3, 21, 1)),      # Depths from 3 to 20 + unlimited depth
            'estimator__estimator__min_samples_split': [2, 5, 10, 20, 50],          # Minimum samples to split
            'estimator__estimator__min_samples_leaf': [1, 2, 5, 10, 20, 50],        # Minimum samples in a leaf
            'estimator__estimator__max_features': ['auto', 'sqrt', 'log2', None],   # Features considered at each split
            'estimator__estimator__max_leaf_nodes': [None] + list(range(10, 101, 10))  # Maximum number of leaf nodes

        }
    ),

    "RandomForest": (
        MultiOutputRegressor(RandomForestRegressor(random_state=seed)),
        {
            'estimator__estimator__n_estimators': [50, 100, 200, 500, 1000],           # Number of trees
            'estimator__estimator__max_depth': [None] + list(range(10, 51, 10)),   # Maximum depth of trees
            'estimator__estimator__min_samples_split': [2, 5, 10, 20, 50],             # Minimum samples to split a node
            'estimator__estimator__min_samples_leaf': [1, 2, 5, 10, 20, 50],          # Minimum samples in a leaf node
            'estimator__estimator__max_features': ['auto', 'sqrt', 'log2', None],  # Features considered for splits
          
        }  
    ),

    "XGBoost": (
        MultiOutputRegressor(XGBRegressor(objective='reg:squarederror', random_state=seed)),
        {
            'estimator__estimator__n_estimators': [100, 200, 500, 1000],          # Number of trees
            'estimator__estimator__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],  # Learning rate
            'estimator__estimator__max_depth': [3, 5, 7, 10, 15, 20],            # Maximum tree depth
            'estimator__estimator__min_child_weight': [1, 5, 10, 20],            # Minimum child weight
            'estimator__estimator__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],        # Subsampling ratio
            'estimator__estimator__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], # Features per tree
            'estimator__estimator__gamma': [0, 0.1, 0.2, 0.5, 1, 5],             # Minimum loss reduction
            'estimator__estimator__reg_alpha': [0, 0.01, 0.1, 1, 10, 100],       # L1 regularization
            'estimator__estimator__reg_lambda': [1, 10, 50, 100]                 # L2 regularization

        }  
    )
}



# -----------------------------
# Run GridSearchCV for all models
# -----------------------------
results = {}

for name, (model, param_grid) in models.items():
    print(f"Training model: {name}")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('estimator', model)
    ])
    
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        scoring=rmse_scorer,
        cv=cv,
        n_iter=50,         # <-- you choose how many random combos to try
        n_jobs=-1,
        verbose=1,
        random_state=1298
    )
    
    search.fit(X_train, y_train)
    results[name] = search

#--------------------------
# Compare Results
#--------------------------
for name, search in results.items():
    print(f"{name}: Best CV RMSE = {-search.best_score_:.3f}")

Training model: MultiTaskLasso
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: SVR
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: KNN
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: DecisionTree
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: RandomForest
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: XGBoost
Fitting 15 folds for each of 50 candidates, totalling 750 fits
MultiTaskLasso: Best CV RMSE = 0.804
SVR: Best CV RMSE = 0.714
KNN: Best CV RMSE = 0.649
DecisionTree: Best CV RMSE = 0.488
RandomForest: Best CV RMSE = 0.340
XGBoost: Best CV RMSE = 0.435


In [12]:
for name, search in results.items():
    print(f"{name} best params:")
    print(search.best_params_)
    print()

MultiTaskLasso best params:
{'estimator__alpha': 0.08}

SVR best params:
{'estimator__estimator__C': 2.847162213648956, 'estimator__estimator__gamma': 'auto', 'estimator__estimator__kernel': 'rbf'}

KNN best params:
{'estimator__estimator__weights': 'uniform', 'estimator__estimator__p': 2, 'estimator__estimator__n_neighbors': 3, 'estimator__estimator__algorithm': 'auto'}

DecisionTree best params:
{'estimator__estimator__min_samples_split': 50, 'estimator__estimator__min_samples_leaf': 1, 'estimator__estimator__max_leaf_nodes': None, 'estimator__estimator__max_features': 'sqrt', 'estimator__estimator__max_depth': None}

RandomForest best params:
{'estimator__estimator__n_estimators': 200, 'estimator__estimator__min_samples_split': 5, 'estimator__estimator__min_samples_leaf': 1, 'estimator__estimator__max_features': 'log2', 'estimator__estimator__max_depth': None}

XGBoost best params:
{'estimator__estimator__subsample': 0.8, 'estimator__estimator__reg_lambda': 1, 'estimator__estimator_

In [14]:
from sklearn.linear_model import MultiTaskLasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

# Define model mapping (must match what was passed into GridSearchCV)
model_classes = {
    "MultiTaskLasso": MultiTaskLasso,
    "SVR": lambda **kwargs: MultiOutputRegressor(SVR(**kwargs)),
    "KNN": lambda **kwargs: MultiOutputRegressor(KNeighborsRegressor(**kwargs)),
    "DecisionTree": lambda **kwargs: MultiOutputRegressor(DecisionTreeRegressor( **kwargs)),
    "RandomForest": lambda **kwargs: MultiOutputRegressor(RandomForestRegressor( **kwargs)),
    "XGBoost": lambda **kwargs: MultiOutputRegressor(XGBRegressor(objective="reg:squarederror", **kwargs))
}

# Collect results
all_metrics = []

for name, search in results.items():
    best_params = search.best_params_
    
    # Flatten parameter dict for instantiation (remove 'estimator__estimator__' prefix)
    flat_params = {
        k.replace("estimator__estimator__", "").replace("estimator__", ""): v
        for k, v in best_params.items()
    }

    metrics = evaluate_model(
        name=name,
        model_class=model_classes[name],
        best_params=flat_params,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )
    all_metrics.append(metrics)


Evaluating MultiTaskLasso...
Evaluating SVR...
Evaluating KNN...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating XGBoost...


In [15]:
import pandas as pd

results_df = pd.DataFrame([{
    "Model": m["model"],
    "R²": m["r2"],
    "RMSE": m["rmse"],
    "MAE": m["mae"],
    "Avg Geodesic Error (km)": m["avg_km_error"],
    "SE Geodesic Error (km)": m["se_km_error"]
} for m in all_metrics])

results_df.to_csv("../tables/raw/raw_best_case_tuned_results.csv", index=False)

results_df



,Model,R²,RMSE,MAE,Avg Geodesic Error (km),SE Geodesic Error (km)
0,MultiTaskLasso,0.448394,0.745991,0.228303,76.467168,89.006972
1,SVR,0.611294,0.630388,0.095733,59.156881,79.260407
2,KNN,0.709883,0.538869,0.015321,36.048914,76.934562
3,DecisionTree,0.810490,0.440076,0.019726,25.324499,64.851450
4,RandomForest,0.884195,0.344840,0.031582,24.924802,48.151837
5,XGBoost,0.858185,0.381605,0.072625,32.150896,50.605773


In [ ]:
## ANTHING BELOW IS OLD

In [ ]:
## WIP
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn import linear_model
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import numpy as np
from xgboost import XGBRegressor
from tqdm import tqdm
import time
from scipy.stats import uniform

def construct_model():
    # the list of regressors to use
    regression_models = [
        linear_model.MultiTaskLasso(),
        #MultiOutputRegressor(SVR()),
        MultiOutputRegressor(KNeighborsRegressor()),
        MultiOutputRegressor(DecisionTreeRegressor()),
        MultiOutputRegressor(RandomForestRegressor()),
        MultiOutputRegressor(XGBRegressor())
    ]

    # Define hyperparameter distributions with refined ranges
    lasso_parameters = {
        'alpha': np.arange(0.00, 1.0, 0.01)
    }

    svr_parameters = {
        'estimator__C': uniform(0.1, 10),  # Uniform distribution between 0.1 and 10
        'estimator__kernel': ['linear', 'rbf', 'poly'],
        'estimator__gamma': ['scale', 'auto'] + list(np.logspace(-3, 3, 50))
    }

    knn_parameters = {
        'n_neighbors': np.arange(2, 50, 1),  # Number of neighbors (2 to 50)
        'weights': ['uniform', 'distance'],  # Weighting method
        'p': [1, 2],  # Manhattan or Euclidean distance
        'algorithm': ['auto']  # Default to 'auto'
    }

    dt_parameters = {
        'max_depth': [None] + list(range(3, 21, 1)),      # Depths from 3 to 20 + unlimited depth
        'min_samples_split': [2, 5, 10, 20, 50],          # Minimum samples to split
        'min_samples_leaf': [1, 2, 5, 10, 20, 50],        # Minimum samples in a leaf
        'max_features': ['auto', 'sqrt', 'log2', None],   # Features considered at each split
        'max_leaf_nodes': [None] + list(range(10, 101, 10))  # Maximum number of leaf nodes
    }

    rf_parameters = {
        'n_estimators': [100, 200, 500, 1000],           # Number of trees
        'max_depth': [None] + list(range(10, 51, 10)),   # Maximum depth of trees
        'min_samples_split': [2, 5, 10, 20],             # Minimum samples to split a node
        'min_samples_leaf': [1, 2, 4, 10, 20],           # Minimum samples in a leaf node
        'max_features': ['auto', 'sqrt', 'log2', None],  # Features considered for splits
        #'bootstrap': [True],                      # Whether to use bootstrapping
        'max_samples': [0.5, 0.75, None]                 # Optional: Samples per tree if bootstrap=True
    }

    xgb_parameters = {
        'estimator__n_estimators': [100, 200, 500, 1000],          # Number of trees
        'estimator__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],  # Learning rate
        'estimator__max_depth': [3, 5, 7, 10, 15, 20],            # Maximum tree depth
        'estimator__min_child_weight': [1, 5, 10, 20],            # Minimum child weight
        'estimator__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],        # Subsampling ratio
        'estimator__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], # Features per tree
        'estimator__gamma': [0, 0.1, 0.2, 0.5, 1, 5],             # Minimum loss reduction
        'estimator__reg_alpha': [0, 0.01, 0.1, 1, 10, 100],       # L1 regularization
        'estimator__reg_lambda': [1, 10, 50, 100]                 # L2 regularization
    }


    # Combine parameter distributions into a list
    parameters = [
        lasso_parameters,
        #svr_parameters,
        knn_parameters,
        dt_parameters,
        rf_parameters,
        xgb_parameters
    ]

    data = {'estimators': []}
    
    # Progress bar setup
    total_models = len(regression_models)
    pbar = tqdm(total=total_models, desc="Optimizing Models", unit="model")
    
    # Iterate through each regressor and use RandomizedSearchCV
    for i, regressor in enumerate(regression_models):
        start_time = time.time()
        print(f"Starting optimization for {regressor.__class__.__name__}...")

        clf = RandomizedSearchCV(
            regressor,
            param_distributions=parameters[i],
            n_iter=20,
            scoring='neg_mean_absolute_error',
            cv=3,
            n_jobs=-1,
            error_score='raise',
            verbose=1,
            random_state=37
        )
        
        clf.fit(X_train, y_train)
        elapsed_time = time.time() - start_time
        
        # Store the fitted model
        data['estimators'].append((regressor.__class__.__name__, clf))
        
        # Print the best parameters found
        print(f"Best parameters for {regressor.__class__.__name__}: {clf.best_params_}")
        
        pbar.set_postfix({"Last Model Time (s)": round(elapsed_time, 2)})
        pbar.update(1)

    pbar.close()
    return data


In [18]:
## WIP
construct_model()

Starting optimization for MultiTaskLasso...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Best parameters for MultiTaskLasso: {'alpha': 0.05}
Starting optimization for KNeighborsRegressor...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Best parameters for KNeighborsRegressor: {'weights': 'distance', 'p': 2, 'n_neighbors': 2, 'algorithm': 'auto'}
Starting optimization for DecisionTreeRegressor...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Best parameters for DecisionTreeRegressor: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_leaf_nodes': 70, 'max_features': 'auto', 'max_depth': 17}
Starting optimization for RandomForestRegressor...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Best parameters for RandomForestRegressor: {'n_estimators': 1000, 'min_samples_split': 20, 'min_samples_leaf': 1, 'max_samples': 0.75, 'max_features': 'auto', 'max_depth': 20}
Starting optimization for MultiOutputRegressor...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Optimizing Models: 100%|██████████| 5/5 [03:04<00:00, 36.92s/model, Last Model Time (s)=132]

Best parameters for MultiOutputRegressor: {'estimator__subsample': 0.6, 'estimator__reg_lambda': 10, 'estimator__reg_alpha': 0, 'estimator__n_estimators': 500, 'estimator__min_child_weight': 1, 'estimator__max_depth': 15, 'estimator__learning_rate': 0.2, 'estimator__gamma': 0.2, 'estimator__colsample_bytree': 1.0}


{'estimators': [('MultiTaskLasso',
   RandomizedSearchCV(cv=3, error_score='raise', estimator=MultiTaskLasso(),
                      n_iter=20, n_jobs=-1,
                      param_distributions={'alpha': array([0.  , 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1 ,
          0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2 , 0.21,
          0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3 , 0.31, 0.32,
          0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42, 0.43,
          0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53, 0.54,
          0.55, 0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64, 0.65,
          0.66, 0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75, 0.76,
          0.77, 0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87,
          0.88, 0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98,
          0.99])},
                      random_state=37, scoring='neg_mean_absolute_error',
                  

In [9]:
construct_model()

Fitting 10 folds for each of 96 candidates, totalling 960 fits


In [ ]:
#data = {}
construct_model()

Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits


In [10]:
for estimator in data['estimators']:
  print('Best Score for %s: %s' % (estimator[0], estimator[1].best_score_))
  print('Best Hyperparameters for %s: %s' % (estimator[0], estimator[1].best_params_))
  print("\n")

Best Score for KNeighborsRegressor: -0.06883451996121953
Best Hyperparameters for KNeighborsRegressor: {'n_neighbors': 2, 'weights': 'distance'}




In [6]:
for estimator in data['estimators']:
  print('Best Score for %s: %s' % (estimator[0], estimator[1].best_score_))
  print('Best Hyperparameters for %s: %s' % (estimator[0], estimator[1].best_params_))
  print("\n")

Best Score for MultiTaskLasso: -0.08677049729288025
Best Hyperparameters for MultiTaskLasso: {'alpha': 0.01}


Best Score for MultiOutputRegressor: -0.07225225150444294
Best Hyperparameters for MultiOutputRegressor: {'estimator__C': 0.01, 'estimator__epsilon': 0.001, 'estimator__kernel': 'linear'}




In [ ]:
construct_model()